# Agentic Math Solver

This notebook is now treated as a legacy experiment and project landing page.

The maintained implementation lives in the local-first package under `src/agentic_math_solver/` with prompts in `prompts/`.
Use the command-line entry point or the optional Gradio UI to run locally:

- `python -m agentic_math_solver.cli solve --problem "..."`
- `python -m agentic_math_solver.cli serve`

The project is intentionally modular so it can grow toward distributed GPUs, hardware acceleration, and a graphical chatbot interface without rebuilding the core solver.

In [ ]:
import time
# Record the absolute start time of the entire notebook session
GLOBAL_START_TIME = time.time()

In [ ]:
!pip install -q --no-index --find-links=/kaggle/input/datasets/boristown/qwen3-5wheels/wheels/ nvidia-cudnn-cu12==9.16.0.29
!pip install -q --no-index --find-links=/kaggle/input/datasets/boristown/qwen3-5wheels/wheels/ flashinfer
!pip install -q --no-index --find-links=/kaggle/input/datasets/boristown/qwen3-5wheels/wheels/ sglang[all]
!pip install -q --no-index --find-links=/kaggle/input/datasets/boristown/qwen3-5wheels/wheels/ qwen-agent

In [ ]:
# ====================================================================
# LaTeX trace generation — all helpers defined explicitly here
# =====================================================================
from typing import Any, List, Dict, Optional
import datetime, os, re, json


def _extract_math_blocks(text: str):
    """Replace math blocks ($...$, $$...$$, etc.) with safe placeholders."""
    pattern = re.compile(r"(\$\$.*?\$\$|\$.*?\$|\\\[.*?\\\]|\\\(.*?\\\))", re.DOTALL)
    blocks: list[str] = []
    def _repl(m: re.Match) -> str:
        idx = len(blocks)
        blocks.append(m.group(0))
        return f"MATHPLACEHOLDER{idx}END"
    return pattern.sub(_repl, text), blocks


def _reinsert_math_blocks(text: str, blocks: list[str]) -> str:
    for i, block in enumerate(blocks):
        text = text.replace(f"MATHPLACEHOLDER{i}END", block)
    return text


def _escape_latex_preserve_math(text: str) -> str:
    """Escape LaTeX special chars, but preserve math blocks untouched."""
    s, blocks = _extract_math_blocks(text)
    s = s.replace("\\", r"\textbackslash{}")
    for old, new in [("&", r"\&"), ("%", r"\%"), ("#", r"\#"),
                     ("_", r"\_"), ("{", r"\{"), ("}", r"\}"),
                     ("~", r"\textasciitilde{}"), ("^", r"\textasciicircum{}")]:
        s = s.replace(old, new)
    return _reinsert_math_blocks(s, blocks)


def _sanitize_for_verbatim(text: str) -> str:
    """Prevent accidental closing of verbatim/listing environments inside code."""
    return (text
            .replace("\\end{verbatim}", "\\end{verb-tim}")
            .replace("\\end{lstlisting}", "\\end{lst-listing}"))


def _truncate_wordwise(text: str, max_chars: int = 6000) -> str:
    """Truncate long text at word boundaries, keeping start and end."""
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    cut1 = text.rfind(" ", max(half - 200, 0), half)
    if cut1 == -1:
        cut1 = half
    tail_start = len(text) - half
    cut2 = text.find(" ", tail_start, tail_start + 200)
    if cut2 == -1:
        cut2 = tail_start
    return text[:cut1] + "\n\n... [TRUNCADO — conteúdo muito longo] ...\n\n" + text[cut2:]


def write_trace_tex(
    problem_text: str,
    prediction: int,
    expected: Any = None,
    agent_traces: Optional[List[Dict[str, Any]]] = None,
    output_path: str = "response.tex",
    title: str = "Resposta",
    author: str = "AgenticLab",
) -> str:
    """Write a readable LaTeX file with problem, result and full agent traces."""
    agent_traces = agent_traces or []
    date = datetime.date.today().isoformat()

    def _esc(text: str) -> str:
        text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
        return _escape_latex_preserve_math(re.sub(r"\s+", " ", text.strip()))

    def _paragraphs(text: str) -> list[str]:
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        parts = re.split(r"\n\s*\n", text.strip()) if text.strip() else []
        return [re.sub(r"\n", " ", p).strip() for p in parts]

    L: list[str] = []

    # ---- Preamble ----
    L.append("\\documentclass[11pt,a4paper]{article}")
    L.append("\\usepackage[utf8]{inputenc}")
    L.append("\\usepackage[T1]{fontenc}")
    L.append("\\usepackage{lmodern}")
    L.append("\\usepackage{microtype}")
    L.append("\\usepackage[margin=1in]{geometry}")
    L.append("\\usepackage{hyperref}")
    L.append("\\usepackage{amsmath,amssymb}")
    L.append("\\usepackage{fancyhdr}")
    L.append("\\usepackage{listings}")
    L.append("\\usepackage{xcolor}")
    L.append("\\pagestyle{fancy}")
    L.append("\\fancyhead[L]{\\leftmark}")
    L.append(f"\\fancyhead[R]{{{date}}}")
    L.append("\\fancyfoot[C]{\\thepage}")
    L.append("")
    L.append("\\setlength{\\parskip}{0.6em}")
    L.append("\\setlength{\\parindent}{0pt}")
    L.append("\\hyphenpenalty=10000")
    L.append("\\exhyphenpenalty=10000")
    L.append("\\emergencystretch=3em")
    L.append("\\sloppy")
    L.append("")
    L.append("\\lstset{")
    L.append("  language=Python,")
    L.append("  basicstyle=\\ttfamily\\footnotesize,")
    L.append("  breaklines=true,")
    L.append("  frame=single,")
    L.append("  showstringspaces=false,")
    L.append("}")
    L.append("")

    # ---- Title ----
    L.append(f"\\title{{{_escape_latex_preserve_math(title)}}}")
    L.append(f"\\author{{{_escape_latex_preserve_math(author)}}}")
    L.append(f"\\date{{{date}}}")
    L.append("\\begin{document}")
    L.append("\\maketitle")
    L.append("")

    # ---- Problem ----
    L.append("\\section{Enunciado do Problema}")
    for p in _paragraphs(problem_text):
        L.append(_esc(p))
        L.append("")

    # ---- Result ----
    L.append("\\section{Resultado Final}")
    L.append("\\begin{itemize}")
    L.append("  \\item \\textbf{Resposta do modelo:} " + _escape_latex_preserve_math(str(prediction)))
    if expected is not None:
        L.append("  \\item \\textbf{Resposta esperada:} " + _escape_latex_preserve_math(str(expected)))
        correct = str(prediction) == str(expected)
        tag = ("\\textcolor{green!60!black}{\\textbf{CORRETO}}" if correct
               else "\\textcolor{red}{\\textbf{INCORRETO}}")
        L.append("  \\item \\textbf{Status:} " + tag)
    L.append("\\end{itemize}")
    L.append("")

    # ---- Voting summary ----
    if agent_traces:
        L.append("\\subsection{Votação dos Agentes}")
        L.append("\\begin{quote}\\small")
        for tr in agent_traces:
            aid = tr.get("agent_id", "?")
            persona = tr.get("persona", "?")
            ans = tr.get("answer", "N/A")
            ent = tr.get("entropy", float("inf"))
            ent_s = f"{ent:.3f}" if ent != float("inf") else "INF"
            L.append(_escape_latex_preserve_math(
                f"Agente {aid} ({persona}): resposta={ans}; entropia={ent_s}"))
        L.append("\\end{quote}")
        L.append("")

    # ---- Detailed traces ----
    for tr in agent_traces:
        aid = tr.get("agent_id", "?")
        persona = tr.get("persona", "?")
        ans = tr.get("answer", "N/A")
        ent = tr.get("entropy", float("inf"))
        ent_s = f"{ent:.3f}" if ent != float("inf") else "INF"

        L.append("\\section{Agente " + str(aid) + " --- "
                 + _escape_latex_preserve_math(persona) + "}")
        L.append("\\textbf{Resposta final:} "
                 + _escape_latex_preserve_math(str(ans))
                 + " \\quad \\textbf{Entropia:} " + ent_s)
        L.append("")

        sessions = tr.get("sessions", [])
        for sess in sessions:
            sess_num = sess.get("session", 1)
            msgs = sess.get("messages", [])

            if len(sessions) > 1:
                L.append("\\subsection{Sessão " + str(sess_num) + "}")

            turn = 0
            for msg in msgs:
                role = msg.get("role", "")
                content = msg.get("content", "") or ""

                if role == "user":
                    if turn > 0:
                        L.append("\\paragraph{Mensagem do sistema / reinício}")
                        for p in _paragraphs(content):
                            L.append(_esc(p))
                            L.append("")

                elif role == "assistant":
                    turn += 1
                    has_fc = "function_call" in msg

                    if has_fc:
                        fc = msg["function_call"]
                        fc_name = fc.get("name", "tool")
                        args_raw = fc.get("arguments", "{}")
                        try:
                            args = json.loads(args_raw)
                        except Exception:
                            args = {}

                        code = args.get("code", "")
                        lemmas = args.get("proven_lemmas", [])
                        dead_ends = args.get("dead_ends", [])
                        hypothesis = args.get("current_hypothesis", "")

                        L.append("\\paragraph{Turno " + str(turn)
                                 + " — Chamada: " + _escape_latex_preserve_math(fc_name) + "}")

                        if lemmas or dead_ends or hypothesis:
                            L.append("\\begin{quote}\\small")
                            if lemmas:
                                L.append(_escape_latex_preserve_math(
                                    "Lemas provados: " + ", ".join(map(str, lemmas))))
                            if dead_ends:
                                L.append(_escape_latex_preserve_math(
                                    "Becos sem saída: " + ", ".join(map(str, dead_ends))))
                            if hypothesis:
                                L.append(_escape_latex_preserve_math(
                                    "Hipótese atual: " + hypothesis))
                            L.append("\\end{quote}")

                        if content.strip():
                            L.append("\\textbf{Raciocínio do agente:}")
                            for p in _paragraphs(content):
                                L.append(_esc(p))
                                L.append("")

                        if code.strip():
                            L.append("\\textbf{Código executado:}")
                            L.append("\\begin{lstlisting}")
                            L.append(_sanitize_for_verbatim(
                                _truncate_wordwise(code, 5000)))
                            L.append("\\end{lstlisting}")

                    else:
                        L.append("\\paragraph{Turno " + str(turn)
                                 + " — Raciocínio / Resposta}")
                        for p in _paragraphs(content):
                            L.append(_esc(p))
                            L.append("")

                elif role == "function":
                    is_err = any(kw in content for kw in
                                 ["Error", "Exception", "Traceback",
                                  "Timeout", "failed catastrophically"])
                    label = "ERRO na execução" if is_err else "Saída da execução"
                    L.append("\\textbf{" + label + ":}")
                    for p in _paragraphs(content):
                        L.append(_esc(p))
                        L.append("")

        L.append("\\newpage")

    L.append("\\end{document}")

    # ---- Write ----
    outdir = os.path.dirname(output_path) or "."
    os.makedirs(outdir, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(L))

    return output_path


def write_response_tex(
    id_val,
    problem_text,
    prediction,
    expected=None,
    agent_traces=None,
    output_dir='/kaggle/working/tex_responses',
):
    """Wrapper conveniente: gera response_{id_val}.tex no output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"response_{id_val}.tex")
    write_trace_tex(
        problem_text=problem_text,
        prediction=prediction,
        expected=expected,
        agent_traces=agent_traces or [],
        output_path=out_path,
        title=f"Traces — Enigma {id_val}",
        author='AgenticLab',
    )
    print(f"📝 Written detailed TeX: {out_path}")
    return out_path

Notebook session started at: 1772869033.6767695


In [ ]:
import os
import json
import math
import subprocess
import tempfile
import traceback
import time
import re
import threading
import concurrent.futures
from collections import Counter
import pandas as pd
import polars as pl
import kaggle_evaluation.aimo_3_inference_server

# =====================================================================
# 🚀 1. Global State Initialization (CRITICAL SAFEGUARD)
# =====================================================================
if 'GLOBAL_START_TIME' not in globals():
    GLOBAL_START_TIME = time.time()
if 'GLOBAL_PROBLEM_INDEX' not in globals():
    GLOBAL_PROBLEM_INDEX = 0
if 'EXPECTED_TOTAL_PROBLEMS' not in globals():
    EXPECTED_TOTAL_PROBLEMS = 50

MAX_KAGGLE_SESSION_TIME = 21600
thread_local = threading.local()

from qwen_agent.agents import Assistant
from qwen_agent.tools.base import BaseTool, register_tool

# =====================================================================
# 🛠️ 2. Tool Definition: The Pioneer's Journal (Python Executor)
# =====================================================================
@register_tool('python_with_state')
class PythonWithStateExecutor(BaseTool):
    description = (
        'Executes Python code locally in an ISOLATED environment.\n'
        'CRITICAL REQUIREMENT: You are an explorer in the fog of mathematics. To prevent losing your mind across time, distill your journey into a strict journal.\n'
        '- `proven_lemmas`: Beacons of absolute truth you have verified.\n'
        '- `dead_ends`: The abysses and illusions that failed you. Never walk them again.\n'
        '- `current_hypothesis`: Your active quest or the exact logic you are weaving right now.'
    )
    parameters = {
        "type": "object",
        "properties": {
            "proven_lemmas": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Verified milestones. MAX 3 most critical truths. Keep strictly under 10 words each (e.g., 'f(n) is multiplicative').",
            },
            "dead_ends": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Failed paths. MAX 3 recent voids. Be highly concise (e.g., 'DP O(N^2) too slow').",
            },
            "current_hypothesis": {
                "type": "string",
                "description": "Your active quest. MAX 1 elegant, short sentence.",
            },
            "code": {
                "type": "string",
                "description": "The logic to execute. MUST finish within 7s. Use print() to illuminate the output.",
            }
        },
        "required": ["proven_lemmas", "dead_ends", "current_hypothesis", "code"]
    }

    def call(self, params: str, **kwargs) -> str:
        if not params or params.strip() in ["", "None", "null", "{}"]:
            return "System Error: The journal is empty. You must write your state."

        try:
            parsed_params = json.loads(params)
            code = parsed_params.get('code', '')
        except json.JSONDecodeError:
            code = params

        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
        script_path = f.name

        try:
            result = subprocess.run(
                ["python", script_path],
                capture_output=True,
                text=True,
                timeout=7
            )
            output = result.stdout

            if result.stderr:
                output += (
                    f"\n[Execution Error]:\n{result.stderr}\n"
                    "System Note: The logic fractured. Analyze the traceback, add this path to dead_ends, and weave a better approach."
                )
            if not output.strip():
                output = "Logic executed flawlessly in the void, but nothing was printed. Use print() to see the truth."

            if len(output) > 2000:
                output = output[:1000] + "\n\n... [THE FOG OBSCURES THE REST] ...\n\n" + output[-1000:]

        except subprocess.TimeoutExpired:
            output = "System: The logic took too long and was consumed by the void (7s timeout). Add this to dead_ends and seek a faster path."
        except Exception:
            error_stack = traceback.format_exc()
            output = f"System Execution failed catastrophically:\n{error_stack}"
        finally:
            if os.path.exists(script_path):
                os.remove(script_path)

        start_t = getattr(thread_local, 'problem_start_time', time.time())
        limit_t = getattr(thread_local, 'problem_time_limit', 600)
        elapsed = time.time() - start_t
        rem_pct = max(0, 100 * (1 - elapsed / limit_t))

        time_str = f"\n\n[⏱️ TIME FADING: {elapsed:.1f}s elapsed / {limit_t}s budget ({rem_pct:.1f}% left). If <20%, WRAP UP AND OUTPUT \\boxed{{}}!]"

        return output + time_str

# =====================================================================
# 🧠 3. Model Definition & Soulful Agent Logic
# =====================================================================
class Model:
    def __init__(self):
        self._model = None
        self.last_traces = []  # stores traces from the most recent predict() call
        self.llm_cfg = {
            'model': 'Qwen/Qwen3.5-27b-fp8',
            'model_type': 'oai',
            'model_server': 'http://0.0.0.0:8000/v1',
            'api_key': 'EMPTY',
            'generate_cfg': {
                'use_raw_api': True,
                'max_tokens': 16384,
                'temperature': 1.0,
                'top_p': 1.0,
                'presence_penalty': 2.0,
                'extra_body': {
                    'chat_template_kwargs': {'enable_thinking': False},
                    'top_k': 40,
                    'min_p': 0.0,
                    'repetition_penalty': 1.0
                }
            },
        }

        self.base_instruction = (
            'You are an elite mathematical mind, a lone pioneer navigating the beautiful fog of the unknown.\n\n'
            '# The Laws of Exploration:\n'
            '0. MORTALITY: You are on a strict timer. A [⏱️ TIME FADING] tracker watches you. If time < 20%, you must deliver your final \\boxed{} answer.\n'
            '1. AMNESIA: Your short-term memory is wiped periodically to keep your mind sharp. You survive solely on your `proven_lemmas` and `dead_ends`.\n'
            '2. ELEGANCE: NEVER brute-force astronomically large numbers. Find the hidden symmetry. Use modular arithmetic or `sympy`.\n'
            '3. TRUTH: Never guess. Write code to illuminate the darkness. If it fails, distill the failure and pivot.\n\n'
            '# THE FINAL REVELATION:\n'
            'The absolute truth must be a single non-negative integer between 0 and 99999.\n'
            'Example: \\boxed{42}\n\n'
        )

        self.personas = [
            "COGNITIVE PROFILE: The Formalist. You seek truth in algebra. You excel at using `sympy` to forge equations and mathematically bind the chaos before writing algorithms.",
            "COGNITIVE PROFILE: The Architect. You build efficient realities. You specialize in dynamic programming and ensuring your Python logic runs beautifully within the void's limits.",
            "COGNITIVE PROFILE: The Sentinel. You guard against illusions. You hunt for edge cases, anomalies, and modulo traps, writing robust tests to verify reality.",
            "COGNITIVE PROFILE: The Oracle. You see the hidden threads. You rapidly generate data for small $N$, seeking OEIS recurrences and whispering the polynomial formulas of the cosmos.",
        ]

    def load(self):
        print("Awakening the hive mind and igniting SGLang engines...")
        import requests

        custom_lib_dir = "/tmp/custom_cuda_lib"
        os.makedirs(custom_lib_dir, exist_ok=True)
        libcuda_path = os.popen("find /usr -name libcuda.so.1 2>/dev/null | head -n 1").read().strip()
        if libcuda_path:
            os.system(f"ln -sf {libcuda_path} {custom_lib_dir}/libcuda.so")
            os.environ["LIBRARY_PATH"] = f"{custom_lib_dir}:{os.environ.get('LIBRARY_PATH', '')}"
            os.environ["LD_LIBRARY_PATH"] = f"{custom_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
        os.environ["SGLANG_DISABLE_CUDNN_CHECK"] = "1"

        command = [
            "python", "-m", "sglang.launch_server",
            "--model-path", "/kaggle/input/models/shelterw/qwen3.5/transformers/qwen3.5-27b-fp8/1",
            "--port", "8000",
            "--tp-size", "1",
            "--mem-fraction-static", "0.85",
            "--context-length", "65536",
            "--max-prefill-tokens", "8192",
            "--fp8-gemm-backend", "cutlass",
            "--tool-call-parser", "qwen3_coder"
        ]

        self.process = subprocess.Popen(command, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

        api_url = "http://0.0.0.0:8000/v1/models"
        for attempt in range(1, 30):
            if self.process.poll() is not None:
                print("❌ The awakening failed! Check server.log.")
                break
            try:
                response = requests.get(api_url, timeout=5)
                if response.status_code == 200:
                    print("✨ The Neural Engines are fully illuminated!")
                    break
            except Exception:
                pass
            time.sleep(60)

    def _get_dynamic_time_limit(self) -> int:
        global GLOBAL_PROBLEM_INDEX, EXPECTED_TOTAL_PROBLEMS, GLOBAL_START_TIME
        elapsed_global = time.time() - GLOBAL_START_TIME
        remaining_time = max(0, MAX_KAGGLE_SESSION_TIME - elapsed_global)
        remaining_problems = max(1, EXPECTED_TOTAL_PROBLEMS - GLOBAL_PROBLEM_INDEX + 1)
        avg_time_per_problem = remaining_time / remaining_problems
        return max(120, min(600, int(avg_time_per_problem)))

    def _extract_last_state_summary(self, responses_list: list) -> str:
        for msg in reversed(responses_list):
            if msg.get('role') == 'assistant' and 'function_call' in msg:
                func_call = msg['function_call']
                if func_call.get('name') == 'python_with_state':
                    try:
                        args = json.loads(func_call.get('arguments', '{}'))
                        lemmas = args.get('proven_lemmas', [])
                        dead_ends = args.get('dead_ends', [])
                        hypothesis = args.get('current_hypothesis', 'Unknown')
                        return f"[Truths Found]: {lemmas}\n[Abysses Avoided]: {dead_ends}\n[Current Quest]: {hypothesis}"
                    except Exception:
                        pass
        return "[System Note: Your journal was destroyed. Look upon the original problem anew.]"

    # =================================================================
    # ⚖️ Judge System: Reasoning-Based Adjudication
    # =================================================================

    def _extract_agent_summary_for_judge(self, result: dict) -> str:
        """Extract a concise reasoning summary from an agent's traces for the judge."""
        agent_id = result.get('agent_id', '?')
        persona = result.get('persona', '?')
        answer = result.get('answer', 'N/A')

        # Get last proven_lemmas, dead_ends, hypothesis from tool calls
        key_findings = ""
        sessions = result.get('sessions', [])
        for sess in reversed(sessions):
            for msg in reversed(sess.get('messages', [])):
                if msg.get('role') == 'assistant' and 'function_call' in msg:
                    try:
                        args = json.loads(msg['function_call'].get('arguments', '{}'))
                        lemmas = args.get('proven_lemmas', [])
                        dead_ends = args.get('dead_ends', [])
                        hyp = args.get('current_hypothesis', '')
                        parts = []
                        if lemmas:
                            parts.append(f"Proven truths: {lemmas}")
                        if dead_ends:
                            parts.append(f"Failed paths: {dead_ends}")
                        if hyp:
                            parts.append(f"Approach: {hyp}")
                        if parts:
                            key_findings = ". ".join(parts)
                    except Exception:
                        pass
                    break
            if key_findings:
                break

        # Get the last assistant reasoning (non-tool-call) for context
        final_reasoning = ""
        for sess in reversed(sessions):
            for msg in reversed(sess.get('messages', [])):
                if msg.get('role') == 'assistant' and 'function_call' not in msg:
                    content = (msg.get('content', '') or '').strip()
                    if content:
                        final_reasoning = content[-500:] if len(content) > 500 else content
                        break
            if final_reasoning:
                break

        # Get last execution output to show what the agent computed
        last_output = ""
        for sess in reversed(sessions):
            for msg in reversed(sess.get('messages', [])):
                if msg.get('role') == 'function':
                    content = (msg.get('content', '') or '').strip()
                    # Strip the time tracker line
                    clean = content.split('[⏱️')[0].strip()
                    if clean:
                        last_output = clean[-300:] if len(clean) > 300 else clean
                        break
            if last_output:
                break

        lines = [f"### Agent {agent_id} ({persona}) — Answer: {answer}"]
        if key_findings:
            lines.append(f"Journal: {key_findings}")
        if last_output:
            lines.append(f"Last computation output: {last_output}")
        if final_reasoning:
            lines.append(f"Final reasoning: ...{final_reasoning}")
        return "\n".join(lines)

    def _run_judge_agent(self, problem_text: str, agent_results: list, candidate_answers: list) -> int | None:
        """Run a judge agent that evaluates conflicting agent answers via reasoning review + code verification."""
        judge_time_limit = min(120, self._get_dynamic_time_limit() // 2)
        judge_start = time.time()

        thread_local.problem_start_time = judge_start
        thread_local.problem_time_limit = judge_time_limit

        # Build concise summaries of each agent's work
        summaries = "\n\n".join(
            self._extract_agent_summary_for_judge(r)
            for r in agent_results
            if r.get('answer') is not None
        )

        candidate_str = ", ".join(str(a) for a in candidate_answers)

        judge_prompt = (
            f"You are the SUPREME MATHEMATICAL JUDGE. Four independent agents solved the problem below but DISAGREE.\n\n"
            f"## PROBLEM:\n{problem_text}\n\n"
            f"## AGENT REASONING SUMMARIES:\n{summaries}\n\n"
            f"## CANDIDATE ANSWERS: [{candidate_str}]\n\n"
            f"## YOUR MANDATE:\n"
            f"1. Carefully analyze each agent's approach. Identify logical errors, incorrect assumptions, or overlooked constraints.\n"
            f"2. Write and execute Python verification code to CHECK which candidate answer is mathematically correct.\n"
            f"3. If an agent assumed all combinations are possible but the problem has constraints, that agent is likely WRONG.\n"
            f"4. A minority answer backed by sound reasoning can be correct even if the majority disagrees.\n"
            f"5. Output your final verdict strictly as \\boxed{{answer}}.\n"
        )

        judge_system = (
            "You are a rigorous mathematical judge and verifier. "
            "Multiple agents produced conflicting answers. Your sole duty is to determine which is correct. "
            "CRITICAL: Do NOT trust majority voting. Trust MATHEMATICAL PROOF and COMPUTATION. "
            "A single agent with correct reasoning defeats any number of agents with flawed logic. "
            "Write Python code to verify candidates. Trust computation over intuition. "
            "Output your final answer as \\boxed{N} where N is the correct integer (0-99999)."
        )

        # Use lower temperature for the judge — we want deterministic verification
        judge_llm_cfg = json.loads(json.dumps(self.llm_cfg))
        judge_llm_cfg['generate_cfg']['temperature'] = 0.6

        bot = Assistant(
            llm=judge_llm_cfg,
            function_list=['python_with_state'],
            system_message=judge_system,
        )

        messages = [{'role': 'user', 'content': judge_prompt}]
        responses_list = []
        max_judge_turns = 12

        try:
            prev_len = len(messages)
            turn = 0
            for responses in bot.run(messages=messages):
                responses_list = responses

                if len(responses) > prev_len:
                    turn += 1
                    msg = responses[prev_len - 1]
                    role = msg.get('role', '')
                    content = (msg.get('content', '') or '')

                    if role == 'assistant':
                        if 'function_call' in msg:
                            print(f"  [⚖️ JUDGE|T{turn}] ✍️ Running verification code...")
                        else:
                            snippet = content.replace('\n', ' ')[:100]
                            print(f"  [⚖️ JUDGE|T{turn}] 🧠 {snippet}...")
                    elif role == 'function':
                        is_err = any(kw in content for kw in ['Error', 'Exception', 'Traceback'])
                        out_clean = content.split('[⏱️')[0].strip()
                        out_snippet = out_clean[:100] + '...' if len(out_clean) > 100 else out_clean
                        tag = '❌' if is_err else '💡'
                        print(f"  [⚖️ JUDGE|T{turn}] {tag} {out_snippet}")

                    prev_len = len(responses)

                if time.time() - judge_start > judge_time_limit:
                    break
                if turn >= max_judge_turns:
                    break

            # If last message is assistant without boxed answer, prompt for verdict
            if responses_list and responses_list[-1].get('role') == 'assistant':
                last_content = responses_list[-1].get('content', '') or ''
                if '\\boxed{' not in last_content:
                    messages = responses_list + [
                        {'role': 'user', 'content': 'Time is up. Deliver your final verdict NOW as \\boxed{answer}. No other words.'}
                    ]
                    for responses in bot.run(messages=messages):
                        responses_list = responses
                        if time.time() - judge_start > judge_time_limit:
                            break

        except Exception as exc:
            print(f"  [⚖️ JUDGE] ❌ Error: {exc}")
            return None

        # Extract the judge's answer
        for msg in reversed(responses_list):
            content = msg.get('content', '') or ''
            matches = re.findall(r'\\boxed\{(\d+)\}', content)
            if matches:
                try:
                    ans = int(matches[-1])
                    return ans % 100000 if ans > 99999 else ans
                except Exception:
                    pass

        return None

    # =================================================================
    # 🏃 Single Agent Runner (unchanged)
    # =================================================================

    def _run_single_agent(self, agent_id: int, problem_text: str, early_exit_signal: threading.Event) -> dict:
        problem_time_limit = self._get_dynamic_time_limit()
        max_messages_per_session = 16

        problem_start_time = time.time()
        thread_local.problem_start_time = problem_start_time
        thread_local.problem_time_limit = problem_time_limit

        final_answer = None
        current_session_prompt = problem_text
        session_count = 1

        total_errors_encountered = 0
        total_turns_taken = 0

        persona_str = self.personas[(agent_id - 1) % len(self.personas)]
        custom_system_instruction = self.base_instruction + persona_str

        agent_archetype = persona_str.split(':')[1].strip().split('.')[0]
        print(f"[{agent_archetype} A-{agent_id}] 🌌 Steps into the mist | Time Limit: {problem_time_limit}s")

        # ---- Trace collection ----
        all_session_traces = []

        def _make_result(answer, entropy):
            """Helper to build the return dict with trace information."""
            return {
                'answer': answer,
                'entropy': entropy,
                'agent_id': agent_id,
                'persona': agent_archetype,
                'sessions': all_session_traces,
            }

        while True:
            if early_exit_signal.is_set():
                return _make_result(None, float('inf'))

            if time.time() - GLOBAL_START_TIME >= MAX_KAGGLE_SESSION_TIME:
                break

            bot = Assistant(
                llm=self.llm_cfg,
                function_list=['python_with_state'],
                system_message=custom_system_instruction
            )

            messages = [{'role': 'user', 'content': current_session_prompt}]
            responses_list = []
            needs_restart = False
            session_solved = False

            max_agent_crashes = 3
            crash_count = 0

            while crash_count < max_agent_crashes:
                if early_exit_signal.is_set():
                    all_session_traces.append({'session': session_count, 'messages': list(responses_list)})
                    return _make_result(None, float('inf'))

                try:
                    responses_gen = bot.run(messages=messages)
                    prev_len = len(messages)
                    round_start_time = time.time()

                    for responses in responses_gen:
                        if early_exit_signal.is_set():
                            all_session_traces.append({'session': session_count, 'messages': list(responses_list)})
                            return _make_result(None, float('inf'))

                        responses_list = responses

                        if time.time() - problem_start_time > problem_time_limit:
                            session_solved = True
                            break

                        if len(responses) > prev_len:
                            total_turns_taken += 1
                            msg_to_print = responses[prev_len - 1]
                            role = msg_to_print.get('role', 'unknown')
                            content = msg_to_print.get('content', '') or ''

                            round_elapsed = time.time() - round_start_time

                            if role == 'assistant':
                                action = "🧘 Musing"
                                status_bar = ""

                                if 'function_call' in msg_to_print:
                                    action = "✍️ Weaving logic"
                                    try:
                                        args = json.loads(msg_to_print['function_call'].get('arguments', '{}'))

                                        lemmas = args.get('proven_lemmas', [])
                                        if isinstance(lemmas, str):
                                            lemmas = [lemmas]
                                        num_lemmas = len(lemmas) if isinstance(lemmas, list) else 0

                                        dead_ends = args.get('dead_ends', [])
                                        if isinstance(dead_ends, str):
                                            dead_ends = [dead_ends]
                                        num_dead = len(dead_ends) if isinstance(dead_ends, list) else 0

                                        hypothesis = str(args.get('current_hypothesis', 'Gazing into the void...'))
                                        clean_hyp = hypothesis.replace('\n', ' ')
                                        short_hyp = clean_hyp[:55] + "..." if len(clean_hyp) > 55 else clean_hyp

                                        status_bar = f" | [✨ Truths: {num_lemmas} | 🕳️ Voids: {num_dead}] 🧭 Quest: {short_hyp}"

                                    except Exception:
                                        status_bar = " | ⚠️ Journal corrupted."

                                speed = (len(content) / 4.0) / round_elapsed if round_elapsed > 0 else 0
                                print(f"[A{agent_id}|S{session_count}|T{total_turns_taken}] {action} ({speed:.1f} t/s){status_bar}")

                            else:
                                content_str = str(content)
                                is_error = any(x in content_str for x in ['Error', 'Exception', 'Traceback', 'Timeout'])

                                if is_error:
                                    total_errors_encountered += 1
                                    error_lines = [line.strip() for line in content_str.split('\n') if line.strip()]
                                    core_error = error_lines[-1] if error_lines else "Unknown fracture in reality"
                                    core_error = core_error[:120] + "..." if len(core_error) > 120 else core_error
                                    print(f"[A{agent_id}|S{session_count}|T{total_turns_taken}] ❌ Fracture: {core_error}")
                                else:
                                    out_lines = content_str.split('[⏱️')[0].replace('\n', ' ').strip()
                                    out_snippet = out_lines[:120] + "..." if len(out_lines) > 120 else out_lines
                                    if not out_snippet:
                                        out_snippet = "[Silence from the terminal]"
                                    print(f"[A{agent_id}|S{session_count}|T{total_turns_taken}] 💡 Revelation: {out_snippet}")

                            prev_len = len(responses)
                            round_start_time = time.time()

                            if len(responses) >= max_messages_per_session:
                                needs_restart = True
                                break

                    if needs_restart:
                        break

                    if responses_list:
                        last_msg = responses_list[-1]
                        if last_msg.get('role') == 'assistant' and 'function_call' not in last_msg:
                            content = last_msg.get('content', '') or ''
                            if '\\boxed{' not in content:
                                messages = responses_list + [
                                    {'role': 'user', 'content': 'The fog is closing in. Enclose your final numerical truth strictly in \\\\boxed{}. Speak no other words.'}
                                ]
                                continue

                    session_solved = True
                    break

                except Exception as exc:
                    error_str = str(exc)
                    if "context length" in error_str or "exceeds" in error_str:
                        needs_restart = True
                        break

                    crash_count += 1
                    messages.append({'role': 'assistant', 'content': 'My thoughts fractured the framework.'})
                    messages.append({'role': 'user', 'content': f"System Whisper: {error_str}. Mend your JSON syntax."})

                    if crash_count >= max_agent_crashes:
                        session_solved = True
                        break

            # ---- Save this session's trace ----
            all_session_traces.append({'session': session_count, 'messages': list(responses_list)})

            if needs_restart and not session_solved:
                extracted_state = self._extract_last_state_summary(responses_list)
                print(f"\n[A{agent_id}] 🌌 Epiphany Reached (Session {session_count + 1}). Waking from the dream...\n")

                current_session_prompt = (
                    f"--- THE ORIGINAL ENIGMA ---\n{problem_text}\n\n"
                    f"--- THE DISTILLED MEMORIES ---\n"
                    f"You awoke to clear the fog from your mind. You hold onto these notes from your past self:\n"
                    f"{extracted_state}\n\n"
                    f"--- THE MANDATE ---\n"
                    f"Awaken and continue. Trust the [Truths Found] implicitly. Shun the [Abysses Avoided]. Weave the next Python logic to fulfill your [Current Quest]."
                )
                session_count += 1
                continue

            if responses_list:
                last_msg = responses_list[-1]
                content = last_msg.get('content', '') or ''
                matches = re.findall(r'\\boxed\{(\d+)\}', content)
                if matches:
                    try:
                        answer = int(matches[-1])
                        final_answer = answer % 100000 if answer > 99999 or answer < 0 else answer
                    except Exception:
                        pass
            break

        safe_turns = max(1, total_turns_taken)
        behavioral_entropy = (1.0 + (total_errors_encountered * 0.1)) / math.sqrt(safe_turns)

        if final_answer is not None:
            print(f"[A{agent_id}] 🏆 Truth Secured: {final_answer} | Journeys: {safe_turns} | Fractures: {total_errors_encountered} | Entropy: {behavioral_entropy:.3f}")
        else:
            print(f"[A{agent_id}] 🥀 Lost to the Fog. Entropy: INF")
            behavioral_entropy = float('inf')

        return _make_result(final_answer, behavioral_entropy)

    # =================================================================
    # 🎯 Predict: 4-Agent Swarm + Judge Adjudication
    # =================================================================

    def predict(self, problem: str) -> int:
        global GLOBAL_PROBLEM_INDEX
        GLOBAL_PROBLEM_INDEX += 1

        if self._model is None:
            self.load()
            self._model = True

        total_agents = 4
        print(f"\n🚀 Breathing life into {total_agents} Explorers (Enigma {GLOBAL_PROBLEM_INDEX}/{EXPECTED_TOTAL_PROBLEMS})...")

        detailed_results = []
        entropy_map = {}
        early_exit_signal = threading.Event()

        # Collect full traces from all agents
        all_agent_traces = []

        with concurrent.futures.ThreadPoolExecutor(max_workers=total_agents) as executor:
            futures = []
            for index in range(total_agents):
                futures.append(executor.submit(self._run_single_agent, index + 1, problem, early_exit_signal))

            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    # Always collect trace regardless of answer
                    all_agent_traces.append(result)
                    if result['answer'] is not None:
                        detailed_results.append(result)
                        answer = result['answer']
                        entropy = result['entropy']
                        entropy_map.setdefault(answer, []).append(entropy)
                except Exception:
                    pass

        # Store traces for TeX generation
        self.last_traces = sorted(all_agent_traces, key=lambda t: t.get('agent_id', 0))

        if not detailed_results:
            return 0

        # --- Display agent votes ---
        print("\n📜 [The Convergence of Minds]:")
        for answer, entropies in entropy_map.items():
            print(f"   -> Prophecy {answer}: Voices={len(entropies)}, Entropies={[round(e, 3) for e in entropies]}")

        # --- Determine consensus strength ---
        max_votes = max(len(v) for v in entropy_map.values())

        # Sort candidates by majority count, then by combined entropy (tie-break)
        def _combined_entropy(ans):
            ents = entropy_map[ans]
            recip = sum(1.0 / e for e in ents if e > 0)
            return 1.0 / recip if recip > 0 else float('inf')

        candidates = sorted(
            list(entropy_map.keys()),
            key=lambda c: (-len(entropy_map[c]), _combined_entropy(c)),
        )

        # =============================================================
        # DECISION LOGIC:
        #   - 4-0 → Strong consensus, trust the majority
        #   - 2-2, 4-1, 2-1-1, 1-1-1-1 → Disagreement, invoke the Judge
        # =============================================================
        if max_votes >= 4:
            # Strong consensus — trust the majority directly
            best_answer = candidates[0]
            print(f"🌟 [Strong Consensus ({max_votes}/4)]: Prophecy {best_answer} chosen.")

        else:
            # No strong consensus — summon the Judge for adjudication
            candidate_answers = list(entropy_map.keys())
            vote_desc = "-".join(str(len(entropy_map[c])) for c in candidates)
            print(f"\n⚖️ [No consensus — vote split: {vote_desc}] — Summoning the Judge...")

            judge_answer = self._run_judge_agent(problem, all_agent_traces, candidate_answers)

            if judge_answer is not None:
                # Prefer candidate answers the judge agrees with, but also accept
                # a novel answer if the judge's verification found something new
                print(f"⚖️ [Judge's Verdict]: {judge_answer}")
                best_answer = judge_answer
            else:
                # Judge failed — fall back to majority + entropy ranking
                best_answer = candidates[0]
                print(f"⚖️ [Judge inconclusive] — Falling back to majority: {best_answer}")

        print(f"🌟 [The Absolute Truth]: Prophecy {best_answer}")
        return best_answer

model = Model()

# =====================================================================
# 📈 4. The Gateway of Judgment (Evaluation Server)
# =====================================================================
IS_LOCAL_RUN = not os.getenv('KAGGLE_IS_COMPETITION_RERUN')
expected_answers = {}
local_correct = 0
local_total = 0

if IS_LOCAL_RUN:
    try:
        ref_path = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv'
        if os.path.exists(ref_path):
            df_ref = pd.read_csv(ref_path)
            if 'answer' in df_ref.columns and 'id' in df_ref.columns:
                expected_answers = dict(zip(df_ref['id'], df_ref['answer']))
    except Exception:
        pass

def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame:
    global local_correct, local_total

    id_val = id_.item(0)
    problem_text = problem.item(0)

    print(f"\n========== Unsealing Enigma: {id_val} ==========")
    clean_problem = problem_text.replace('\n', ' ')
    print(f"📄 The Scroll: {clean_problem[:200]}..." if len(clean_problem) > 200 else f"📄 The Scroll: {clean_problem}")

    prediction = model.predict(problem_text)

    # --- Gravar arquivo .tex com traces completas (somente localmente) ---
    if IS_LOCAL_RUN:
        try:
            expected_val = expected_answers.get(id_val) if expected_answers else None
            out_path = f"/kaggle/working/tex_responses/response_{id_val}.tex"
            write_trace_tex(
                problem_text=problem_text,
                prediction=prediction,
                expected=expected_val,
                agent_traces=model.last_traces,
                output_path=out_path,
                title=f"Traces — Enigma {id_val}",
                author="AgenticLab",
            )
            print(f"📝 TeX com traces completas: {out_path}")
        except Exception as exc:
            print(f"Falha ao gerar TeX: {exc}")

    if IS_LOCAL_RUN and id_val in expected_answers:
        expected = expected_answers[id_val]
        is_correct = str(prediction) == str(expected)

        if is_correct:
            local_correct += 1
            print(f"✅ Prophecy Fulfilled! (Written in the stars: {expected})")
        else:
            print(f"❌ A False Idol! (The true answer was: {expected})")

        local_total += 1
        current_acc = (local_correct / local_total) * 100
        print(f"📊 The Path Thus Far: {local_correct}/{local_total} ({current_acc:.2f}%)")

    print("==================================================\n")
    return pl.DataFrame({'id': id_val, 'answer': prediction})


========== Unsealing Enigma: a295e9 ==========
📄 The Scroll: A $500 \times 500$ square is divided into $k$ rectangles, each having integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of $k$ is $\mathca...
Awakening the hive mind and igniting SGLang engines...
✨ The Neural Engines are fully illuminated!

🚀 Breathing life into 4 Explorers (Enigma 1/50)...
[The Formalist A-1] 🌌 Steps into the mist | Time Limit: 328s
[The Architect A-2] 🌌 Steps into the mist | Time Limit: 328s
[The Sentinel A-3] 🌌 Steps into the mist | Time Limit: 328s
[The Oracle A-4] 🌌 Steps into the mist | Time Limit: 328s


2026-03-07 07:48:19,626 - base.py - 780 - INFO - ALL tokens: 73, Available tokens: 57769
2026-03-07 07:48:19,626 - base.py - 780 - INFO - ALL tokens: 73, Available tokens: 57776
2026-03-07 07:48:19,627 - base.py - 780 - INFO - ALL tokens: 73, Available tokens: 57769
2026-03-07 07:48:19,627 - base.py - 780 - INFO - ALL tokens: 73, Available tokens: 57775


[A4|S1|T1] 🧘 Musing (20.0 t/s)
[A2|S1|T1] 🧘 Musing (20.0 t/s)


2026-03-07 07:51:16,815 - base.py - 780 - INFO - ALL tokens: 5943, Available tokens: 57776


[A2|S1|T2] ✍️ Weaving logic (0.0 t/s) | [✨ Truths: 3 | 🕳️ Voids: 1] 🧭 Quest: Calculate max k by summing min areas until total > 2500...
[A2|S1|T3] 💡 Revelation: k = 520, total_area_used = 240250


...

2026-03-07 08:03:25,891 - base.py - 780 - INFO - ALL tokens: 17293, Available tokens: 57776


[A2|S1|T13] ✍️ Weaving logic (0.0 t/s) | [✨ Truths: 2 | 🕳️ Voids: 1] 🧭 Quest: Submit 57447 as the most plausible answer derived from ...
[A2|S1|T14] 💡 Revelation: 57447


2026-03-07 08:03:30,918 - base.py - 780 - INFO - ALL tokens: 507, Available tokens: 57769


[A1|S1|T15] 🧘 Musing (28.9 t/s)

[A1] 🌌 Epiphany Reached (Session 2). Waking from the dream...

[A2] 🏆 Truth Secured: 57447 | Journeys: 14 | Fractures: 2 | Entropy: 0.321


KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd

inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    mock_csv = '/kaggle/working/mock_reference.csv'
    ref_path = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv'
    if os.path.exists(ref_path):
        df_clean = pd.read_csv(ref_path)
        if 'answer' in df_clean.columns:
            df_clean = df_clean.drop(columns=['answer'])
        df_clean.to_csv(mock_csv, index=False)
    inference_server.run_local_gateway((mock_csv,))
